## HDBSCAN Clustering

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) is a density-based clustering algorithm that identifies clusters based on regions of high data density.

Unlike K-Means, HDBSCAN does not require specifying the number of clusters in advance. Instead, it automatically detects clusters of varying shapes and sizes, while labeling low-density points as **noise**.

### How HDBSCAN Works

The algorithm follows these key steps:

1. Estimate the **local density** of each data point
2. Build a hierarchy of clusters based on density connectivity
3. Extract the most stable clusters from the hierarchy
4. Label points that do not belong to any dense region as **noise**

### Objective Function

HDBSCAN does not optimize a single global objective like K-Means. Instead, it focuses on:

- identifying **dense regions** in the data
- separating them from **sparse regions (noise)**
- selecting clusters that are stable across different density levels

### Key Characteristics

- Produces **soft clustering with noise detection** (points can be labeled as -1 if they do not belong to any cluster)
- Does **not require pre-specifying the number of clusters**
- Can detect:
  - clusters of varying shapes
  - clusters of different densities

### Limitations

- Sensitive to hyperparameters:
  - `min_cluster_size`
  - `min_samples`
- May label a large portion of data as noise if clusters are not well-defined
- Can struggle in very high-dimensional spaces

### Application in This Project

In this project, HDBSCAN is used to evaluate whether the embedding space contains **well-defined dense clusters** of news articles.

If meaningful clusters exist, HDBSCAN should identify them. However, if most points are labeled as noise, this suggests that the data is not organized into clearly separated density regions, but instead forms **continuous semantic gradients**.

In [ ]:
import hdbscan

# Test different HDBSCAN settings
# min_cluster_size = smallest group size required to form a cluster
# min_samples = how conservative the algorithm is when deciding whether a point belongs to a dense region
for mcs in [30, 50, 100]:
    for ms in [5, 10, 20]:
        hdb = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms
        )
        labels = hdb.fit_predict(X_embed_norm)

        # Count the number of clusters, excluding noise label (-1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Proportion of points labeled as noise
        noise_ratio = (labels == -1).mean()
        print(f"min_cluster_size={mcs}, min_samples={ms}, clusters={n_clusters}, noise_ratio={noise_ratio:.3f}")

#min_cluster_size     min_samples      n_clusters      noise_ratio
# 30                  5                2               0.5498333333333333
# 30                  10               0               1.0
# 30                  20               0               1.0
# 50                  5                0               1.0
# 50                  10               0               1.0
# 50                  20               0               1.0
# 100                 5                0               1.0
# 100                 10               0               1.0
# 100                 20               0               1.0

# Density-based clustering (HDBSCAN) identified only one major dense semantic region and labeled the majority of documents as noise,
# suggesting that embedding space is not characterized by sharply separated density cores but rather by continuous semantic gradients
# meaning that changes are smoothly across the embedding space rather than forming sharply separated islands.
# In contrast, partition-based clustering (KMeans) yielded stable and interpretable cluster partitions.

### Results

- Density-based clustering (HDBSCAN) identified only one major dense semantic region and labeled the majority of documents as noise, suggesting that embedding space is not characterized by sharply separated density cores but rather by continuous semantic gradients meaning that changes are smoothly across the embedding space rather than forming sharply separated islands. In contrast, partition-based clustering (KMeans) yielded stable and interpretable cluster partitions.

### Kmeans vs. HDBSCAN
- The contrast between K-Means and HDBSCAN highlights an important property of the data. While K-Means can impose structure by partitioning the space into clusters, HDBSCAN fails to identify stable dense regions, suggesting that the underlying semantic structure is continuous rather than discretely clustered.
